# 11 · Modules, Packages and a Standard-Library Tour

After this notebook you can explain what `import` actually does, lay out a package, talk confidently about virtual environments and version pinning, and reach for the right standard-library tool — `Counter`, `deque`, `lru_cache`, `itertools`, `re`, `datetime`, `hashlib` and friends — instead of writing it yourself in a live-coding round.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [10 · Files, JSON, CSV and pathlib](10_files_json_csv_pathlib.ipynb)

## Why this matters in interviews

- In live coding, the standard library is your speed: `Counter`, `defaultdict`, `deque` and `lru_cache` turn a 15-line answer into three lines, and interviewers read that as experience.
- "How would you structure this service?" leads straight to packages, imports, `__main__`, and why a file called `random.py` broke everything.
- Reproducibility questions ("the result from three weeks ago changed") always include "did you pin your dependencies?" AI SDKs change fast — the `openai` 0.x → 1.x rewrite and LangChain's 1.x reorganisation broke many projects.

## What interviewers ask

- What happens, step by step, when you write `import x`? Where does Python look, and what is `sys.modules`?
- What does `if __name__ == "__main__":` do, and why does it matter for tests and multiprocessing?
- Absolute vs relative imports; what `__init__.py` is for; how you fix a circular import.
- venv vs conda vs uv/poetry; `pip freeze` vs a lock file; what `~=` means.
- "Count the most common labels", "keep the last N messages", "cache this expensive call", "batch these texts", "pull the confidence score out of this model reply with a regex".
- Naive vs aware datetimes — why you store UTC. Why `hash()` must never be a persistent cache key.

## 1. How `import` works

**In plain English:** `import mytools` means "find a file or folder called `mytools`, run its code **once**, keep the resulting module object in a cache, and give me a name for it". Step by step:

1. **Cache check** — if `"mytools"` is already in `sys.modules`, reuse it. Nothing runs again.
2. **Find** — search each folder in `sys.path` in order: the script's folder (or the current folder for `-c` and notebooks), `PYTHONPATH`, the standard library, then `site-packages`.
3. **Load** — run the module's top-level code; for a package (a folder), that means its `__init__.py`.
4. **Bind** — create the name `mytools` in your namespace.

Let's build a tiny package in a temp folder and watch this happen.

In [ ]:
import importlib
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

ROOT = Path(tempfile.mkdtemp(prefix="nb11_"))       # scratch folder, never the repo
pkg = ROOT / "mytools"
pkg.mkdir()
(pkg / "__init__.py").write_text(textwrap.dedent('''
    """mytools: a tiny package for this notebook."""
    print("  [running mytools/__init__.py]")
    from .text import slugify              # relative import: the text module in this package
    __all__ = ["slugify"]
    __version__ = "0.1.0"
'''), encoding="utf-8")
(pkg / "text.py").write_text(textwrap.dedent('''
    import re
    print("  [running mytools/text.py]")

    def slugify(title):
        return re.sub(r"[^a-z0-9]+", "-", title.lower()).strip("-")
'''), encoding="utf-8")
(pkg / "registry.py").write_text(textwrap.dedent('''
    from .text import slugify
    TOOLS = {}

    def register(name):
        TOOLS[slugify(name)] = name

    if __name__ == "__main__":
        register("Search Docs")
        print(TOOLS)
'''), encoding="utf-8")
print(sorted(p.relative_to(ROOT).as_posix() for p in ROOT.rglob("*.py")))

**Predict:** how many `[running ...]` lines appear, and does the second `import` print anything?

In [ ]:
sys.path.insert(0, str(ROOT))                      # make the folder importable
print("first import:")
import mytools
print("second import:")
import mytools                                     # cached in sys.modules: nothing runs
print("slugify:", mytools.slugify("Hello, RAG World!"), "| version:", mytools.__version__)
print("cached modules:", sorted(m for m in sys.modules if m.startswith("mytools")))
print("loaded from:", Path(mytools.__file__).relative_to(ROOT).as_posix())
print("reload runs __init__ again:")
importlib.reload(mytools)
assert "mytools.text" in sys.modules

(If you re-run that cell, even the first import prints nothing — the module is already cached in this kernel. That is also why editing a module file does not take effect in a running notebook until you `importlib.reload` it or restart the kernel.)

Where do everyday modules come from? And a classic trap: a file of **yours** named like a standard module **shadows** it, because the current folder comes first in `sys.path`. A `random.py` next to your script breaks every library that imports `random`.

In [ ]:
import json
import numpy
for mod in (json, numpy):
    print(f"{mod.__name__:<6} -> .../{'/'.join(Path(mod.__file__).parts[-3:])}")
print("sys is compiled into the interpreter:", "sys" in sys.builtin_module_names)

shadow = ROOT / "shadow_demo"
shadow.mkdir()
(shadow / "random.py").write_text('print("I am NOT the real random module")\n', encoding="utf-8")
out = subprocess.run([sys.executable, "-c", "import random; print('has random.choice?', hasattr(random, 'choice'))"],
                     cwd=shadow, capture_output=True, text=True).stdout
print("\nrunning `import random` next to a file called random.py:\n" + out)
assert "has random.choice? False" in out

## 2. `if __name__ == "__main__":`

Every module has a `__name__`. When you **run** a file (`python run_me.py`), its `__name__` is `"__main__"`; when you **import** it, `__name__` is the module's name. The guard lets one file be both a script and an importable module:

- importing it (in a test, or from another module) should **not** start the program;
- `multiprocessing` on Windows and macOS starts workers by *re-importing* your main module — without the guard, each worker would start more workers.

In [ ]:
script = ROOT / "run_me.py"
script.write_text(textwrap.dedent('''
    def main():
        print("main() ran")

    print(f"__name__ is {__name__!r}")
    if __name__ == "__main__":
        main()
'''), encoding="utf-8")

as_script = subprocess.run([sys.executable, str(script)], capture_output=True, text=True).stdout
as_import = subprocess.run([sys.executable, "-c", "import run_me"], cwd=ROOT, capture_output=True, text=True).stdout
print("python run_me.py ->", as_script.strip().replace("\n", " | "))
print("import run_me    ->", as_import.strip().replace("\n", " | "))
assert "main() ran" in as_script and "main() ran" not in as_import

## 3. Packages, `__init__.py`, absolute vs relative imports

A **package** is a folder of modules; its `__init__.py` runs on first import and usually re-exports the public API (`from .text import slugify`), so users write `from mytools import slugify`.

| | Absolute: `from mytools.text import slugify` | Relative: `from .text import slugify` |
|---|---|---|
| Works in | any module, any script | only inside a package |
| Readability | explicit; PEP 8's default recommendation | shorter; survives renaming the package |
| Fails when | the package is not on `sys.path` | you run the file directly as a script |

That last row is a frequent "why does this not work?" moment:

In [ ]:
direct = subprocess.run([sys.executable, str(pkg / "registry.py")], capture_output=True, text=True)
as_module = subprocess.run([sys.executable, "-m", "mytools.registry"], cwd=ROOT, capture_output=True, text=True)
print("python mytools/registry.py ->", direct.stderr.strip().splitlines()[-1])
print("python -m mytools.registry ->", as_module.stdout.strip().splitlines()[-1])
assert "relative import" in direct.stderr

`python -m package.module` runs a module **as part of its package**, so relative imports work. Two more things interviewers ask:

- **Circular imports** (`a` imports `b`, `b` imports `a`) fail with "partially initialized module". Fixes, best first: move the shared code into a third module; import the module rather than names (`import a` then `a.f()` at call time); as a last resort, import inside the function that needs it.
- **A typical AI-service layout** — the `src/` layout stops tests accidentally importing the local folder instead of the installed package:

```
rag-service/
├── pyproject.toml          # dependencies + tool config (ruff, pytest, mypy)
├── uv.lock                 # or requirements.txt: exact pinned versions
├── src/rag_service/
│   ├── __init__.py
│   ├── config.py           # settings from environment variables
│   ├── ingest.py  retrieve.py  generate.py
│   └── api.py              # FastAPI app
├── tests/test_retrieve.py
└── notebooks/
```

## 4. Virtual environments, pip and pinning (explained — nothing is installed here)

**In plain English:** a virtual environment is a private copy of `site-packages` for one project, so project A can use `langchain 0.3` while project B uses `1.4`. The usual commands:

```bash
python -m venv .venv                   # create
.venv\Scripts\activate                 # Windows   (macOS/Linux: source .venv/bin/activate)
python -m pip install "openai>=1.40,<2" pydantic
python -m pip freeze > requirements.txt   # exact versions of everything installed
python -m pip install -r requirements.txt # reproduce it elsewhere
```

| Specifier | Means | Use for |
|---|---|---|
| `openai==1.52.0` | exactly this version | applications, deploys |
| `openai>=1.40,<2` | a range | libraries (don't over-constrain your users) |
| `pydantic~=2.7` | `>=2.7, <3` (compatible release) | "any 2.x from 2.7 on" |
| lock file (`uv.lock`, `poetry.lock`) | every transitive dependency pinned, with hashes | reproducible builds |

`pyproject.toml` *declares* what you need; a lock file *freezes* what you got. `uv` and `poetry` manage both; plain `pip` + `requirements.txt` still works everywhere. From inside Python you can check where you are and what is installed:

In [ ]:
import importlib.metadata as metadata

print("inside a virtual environment:", sys.prefix != sys.base_prefix)
print("python:", sys.version.split()[0])
for name in ["openai", "pydantic", "numpy", "httpx"]:
    print(f"  {name}=={metadata.version(name)}")

## 5. Standard-library tour

One practical example each, AI-engineering flavoured. The first four sections are the ones live-coding rounds lean on hardest.

### 5.1 `collections` — `Counter`, `defaultdict`, `namedtuple`, `deque`, `OrderedDict`

In [ ]:
from collections import Counter, OrderedDict, defaultdict, deque, namedtuple

labels = ["billing", "shipping", "billing", "refund", "billing", "shipping"]
print("Counter     :", Counter(labels).most_common(2))                  # label distribution in one line

chunks = [("refunds.md", "Refunds within 30 days."), ("shipping.md", "3-5 business days."),
          ("refunds.md", "No refunds on digital goods.")]
by_source = defaultdict(list)                                           # missing key -> a fresh []
for source, text in chunks:
    by_source[source].append(text)
print("defaultdict :", dict(by_source))

Hit = namedtuple("Hit", "doc_id score")                                 # a lightweight immutable record
hits = [Hit("refunds.md", 0.82), Hit("shipping.md", 0.41)]
best = max(hits, key=lambda h: h.score)
print("namedtuple  :", best, "-> best.doc_id =", best.doc_id)

history = deque(maxlen=4)                                               # keeps only the last 4 turns
for turn in ["user: hi", "bot: hello", "user: refunds?", "bot: 30 days", "user: digital?", "bot: no"]:
    history.append(turn)
print("deque       :", list(history))
assert Counter(labels)["billing"] == 3 and len(history) == 4 and history[0] == "user: refunds?"

`deque(maxlen=N)` is a sliding window of conversation memory; it also gives O(1) `popleft()`, which a list does not (BFS queues). `OrderedDict.move_to_end` + `popitem(last=False)` is the textbook **LRU cache** — a favourite interview question. (Plain `dict` has kept insertion order since Python 3.7; `OrderedDict` is still the tool when you need to *reorder*.)

In [ ]:
class LRUCache:
    def __init__(self, capacity):
        self.capacity, self.data = capacity, OrderedDict()

    def get(self, key):
        if key not in self.data:
            return None
        self.data.move_to_end(key)                 # mark as most recently used
        return self.data[key]

    def put(self, key, value):
        self.data[key] = value
        self.data.move_to_end(key)
        if len(self.data) > self.capacity:
            self.data.popitem(last=False)          # evict the least recently used

cache = LRUCache(capacity=2)
cache.put("q1", "a1"); cache.put("q2", "a2")
cache.get("q1")                                    # q1 is now the most recent
cache.put("q3", "a3")                              # evicts q2, not q1
print("LRU keeps:", list(cache.data))
assert list(cache.data) == ["q1", "q3"]

### 5.2 `itertools` — lazy building blocks

In [ ]:
from itertools import accumulate, batched, chain, combinations, count, groupby, islice, product

texts = [f"doc{i}" for i in range(7)]
print("batched     :", list(batched(texts, 3)))               # embed in batches of 3 (Python 3.12+)
print("chain       :", list(chain.from_iterable([["a", "b"], ["c"], ["d"]])))   # flatten one level
print("islice+count:", list(islice(count(start=100, step=10), 4)))             # first 4 of an infinite stream
print("product     :", list(product([0.0, 0.7], ["mini", "nano"])))            # a parameter grid
print("combinations:", list(combinations(["A", "B", "C"], 2)))                 # every pair, for pairwise judging
print("accumulate  :", list(accumulate([120, 340, 80, 900])))                  # running token count of a chat

events = [("billing", 1), ("shipping", 2), ("billing", 3)]
print("groupby, unsorted:", [k for k, _ in groupby(events, key=lambda e: e[0])], "<- 'billing' twice!")
events.sort(key=lambda e: e[0])
print("groupby, sorted  :", {k: [v for _, v in g] for k, g in groupby(events, key=lambda e: e[0])})
assert [len(b) for b in batched(texts, 3)] == [3, 3, 1]

`groupby` only groups **consecutive** equal keys — sort first (or use a `defaultdict(list)`). `batched` needs Python 3.12; on older versions, `[texts[i:i + n] for i in range(0, len(texts), n)]`.

### 5.3 `functools` — `lru_cache`, `cache`, `partial`, `reduce`, `wraps`

In [ ]:
from functools import lru_cache, partial, reduce

real_calls = 0

@lru_cache(maxsize=256)
def embed(text):
    global real_calls
    real_calls += 1                                  # stands in for a slow, paid embedding call
    return (len(text), text.count(" "))

for t in ["hello world", "refund policy", "hello world", "hello world"]:
    embed(t)
print("real calls:", real_calls, "|", embed.cache_info())

def call_model(prompt, *, model, temperature):
    return f"[{model} @ T={temperature}] {prompt}"

classify = partial(call_model, model="gpt-4.1-nano", temperature=0)      # pre-fill arguments
print("partial   :", classify("Is 'where is my parcel?' about shipping?"))

layers = [{"model": "gpt-4.1-mini", "temperature": 0.7}, {"temperature": 0.2}, {"max_tokens": 256}]
print("reduce    :", reduce(lambda acc, d: {**acc, **d}, layers, {}))   # merge configs, later wins
assert real_calls == 2 and embed.cache_info().hits == 2

`@cache` is `lru_cache(maxsize=None)` (unbounded). Cached functions must be **pure** and take **hashable** arguments (no lists or dicts). `functools.wraps` is how decorators keep a function's name and docstring — [notebook 12](12_decorators_and_context_managers.ipynb) builds on it.

### 5.4 `re` — pull fields out of model output

A model reply that was *supposed* to follow a format. `search` finds the first match, `findall` all of them, `finditer` gives match objects (with **named groups**), `sub` replaces — here, redacting an email before logging it.

In [ ]:
import re

reply = """Answer: Refunds are issued within 30 days.
Confidence: 0.87
Sources: [doc-12], [doc-7]
Contact: Jane.Doe@Example.com about order ORD-123456"""

confidence = float(re.search(r"Confidence:\s*([0-9.]+)", reply).group(1))
sources = re.findall(r"\[(doc-\d+)\]", reply)
fields = {m["key"].lower(): m["value"] for m in re.finditer(r"^(?P<key>\w+):\s*(?P<value>.+)$", reply, re.MULTILINE)}
redacted = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", "<EMAIL>", reply)
order_ok = bool(re.fullmatch(r"ORD-\d{6}", "ORD-123456"))

print("confidence:", confidence, "| sources:", sources)
print("fields    :", list(fields), "| answer:", fields["answer"])
print("redacted  :", redacted.splitlines()[-1])
print("greedy    :", re.findall(r"\[.*\]", "see [doc-12] and [doc-7]"))
print("lazy      :", re.findall(r"\[.*?\]", "see [doc-12] and [doc-7]"))
print("ignorecase:", bool(re.search(r"^answer:", reply, re.IGNORECASE | re.MULTILINE)), "| fullmatch:", order_ok)
assert confidence == 0.87 and sources == ["doc-12", "doc-7"] and "<EMAIL>" in redacted

Flags worth knowing: `re.MULTILINE` makes `^`/`$` match at every line, `re.DOTALL` lets `.` cross newlines (needed to grab a multi-line JSON block between fences), `re.IGNORECASE`. Always use raw strings `r"..."` so backslashes reach the regex engine, and `re.compile(pattern)` once if you use a pattern in a loop. `.*` is **greedy** (grabs as much as it can); `.*?` is lazy.

### 5.5 `math`, `statistics`, `random` — latency percentiles and reproducible sampling

LLM latency has a long right tail, so the mean hides the pain. `statistics.quantiles(data, n=100)` gives the 99 percentile cut points; index 94 is p95.

In [ ]:
import math
import random
import statistics

import matplotlib.pyplot as plt

BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "axes.grid.axis": "y", "grid.color": "#e1e0d9", "axes.axisbelow": True})

rng = random.Random(42)                                   # an isolated, seeded generator
latencies = [rng.lognormvariate(math.log(800), 0.5) for _ in range(2000)]   # milliseconds
p50 = statistics.median(latencies)
cuts = statistics.quantiles(latencies, n=100)
p95, p99 = cuts[94], cuts[98]
print(f"mean {statistics.fmean(latencies):.0f} ms | p50 {p50:.0f} | p95 {p95:.0f} | p99 {p99:.0f} | "
      f"stdev {statistics.stdev(latencies):.0f}")

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.hist(latencies, bins=60, color=BLUE, alpha=0.85)
for value, name, color in [(p50, "p50", AQUA), (p95, "p95", ORANGE), (p99, "p99", "#52514e")]:
    ax.axvline(value, color=color, lw=2)
    ax.text(value, ax.get_ylim()[1] * 0.92, f" {name} {value:.0f} ms", color="#0b0b0b", fontsize=9)
ax.set_xlabel("simulated LLM request latency (ms)"); ax.set_ylabel("requests")
ax.set_title("2,000 requests: a long right tail, so report p50 and p95, not the mean")
plt.show()
assert statistics.fmean(latencies) > p50 and p99 > p95 > p50

In [ ]:
logprobs = [-0.05, -0.4, -2.3, -0.1]                         # per-token log-probabilities from a model
print("perplexity   :", round(math.exp(-statistics.fmean(logprobs)), 3))
print("batches      :", math.ceil(23 / 8), "for 23 texts in batches of 8")
print("isclose      :", 0.1 + 0.2 == 0.3, math.isclose(0.1 + 0.2, 0.3))

pool = [f"example_{i}" for i in range(10)]
print("sample       :", rng.sample(pool, k=3))                  # few-shot examples, no repeats
split = rng.choices(["model_A", "model_B"], weights=[90, 10], k=1000)
print("weighted     :", Counter(split))                         # a 90/10 traffic split
deck = [1, 2, 3, 4]
print("shuffle      :", rng.shuffle(deck), deck, "<- shuffles in place and returns None")
import secrets
print("secrets token:", len(secrets.token_urlsafe(32)), "chars (use secrets, not random, for keys and tokens)")

`random.Random(seed)` gives you your **own** generator, so seeding it does not depend on (or disturb) any other code that uses the global `random`. For tokens, API keys and anything security-related use `secrets`.

### 5.6 `datetime` + `zoneinfo` — naive vs aware

A **naive** datetime has no timezone: "9:00" — but where? An **aware** one pins a real instant. Rule: store and compare **aware UTC**; convert to a local zone only for display. `datetime.utcnow()` is deprecated since 3.12 (it returns a *naive* value); use `datetime.now(timezone.utc)`.

In [ ]:
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

naive = datetime(2026, 9, 29, 9, 0)
aware = datetime(2026, 9, 29, 9, 0, tzinfo=timezone.utc)
print("naive      :", naive.isoformat(), "| tzinfo =", naive.tzinfo)
print("aware (UTC):", aware.isoformat())
print("Kolkata    :", aware.astimezone(ZoneInfo("Asia/Kolkata")).isoformat())
print("New York   :", aware.astimezone(ZoneInfo("America/New_York")).isoformat())
try:
    naive < aware
except TypeError as e:
    print("naive < aware ->", e)

cached_at = datetime(2026, 9, 29, 8, 0, tzinfo=timezone.utc)
now = datetime(2026, 9, 29, 9, 30, tzinfo=timezone.utc)          # fixed "now" so the output is stable
print("cache age  :", now - cached_at, "| expired (TTL 1h)?", now - cached_at > timedelta(hours=1))
print("parse ISO  :", datetime.fromisoformat("2026-09-29T14:30:00+05:30").astimezone(timezone.utc))

### 5.7 `time` and `timeit` — measure the right thing

| Clock | Use it for |
|---|---|
| `time.perf_counter()` | measuring how long something took (high resolution, never goes backwards) |
| `time.monotonic()` | deadlines and timeouts |
| `time.time()` | wall-clock timestamps for logs (can jump when the system clock is adjusted) |
| `timeit.repeat(...)` | micro-benchmarks: runs many times, take the **minimum** |

In [ ]:
import time
import timeit

start = time.perf_counter()
sum(i * i for i in range(200_000))
print(f"perf_counter: that took {(time.perf_counter() - start) * 1000:.1f} ms")

stop_list = [f"word{i}" for i in range(1000)]
stop_set = set(stop_list)
n = 2000
t_list = min(timeit.repeat(lambda: "word999" in stop_list, number=n, repeat=5)) / n
t_set = min(timeit.repeat(lambda: "word999" in stop_set, number=n, repeat=5)) / n
print(f"'in' a 1,000-item list: {t_list * 1e9:7,.0f} ns | 'in' a set: {t_set * 1e9:5,.0f} ns  (O(n) vs O(1))")
assert t_set < t_list

### 5.8 `os`, `sys` and `logging`

Configuration comes from **environment variables** (never hard-code keys); diagnostics go through **logging**, not `print`: levels let you turn detail up or down without editing code, and handlers send the same records to the console, a file or a log service.

In [ ]:
import logging
import os

os.environ["NB11_MODEL"] = "gpt-4.1-mini"                 # visible to this process and its children only
print("model      :", os.environ.get("NB11_MODEL"), "| with default:", os.getenv("NB11_TEMPERATURE", "0.2"))
print("key present:", "OPENAI_API_KEY" in os.environ, "(check presence; never print a key)")
print("python 3.11+?", sys.version_info >= (3, 11), "| platform:", sys.platform)

log = logging.getLogger("rag.retriever")
log.handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(levelname)-7s %(name)s | %(message)s"))
log.addHandler(handler)
log.propagate = False
log.setLevel(logging.INFO)
log.debug("query vector: %s", [0.12, -0.3])               # below INFO: dropped
log.info("retrieved k=%d chunks in %d ms", 5, 42)         # %-style args: formatted only if emitted
log.warning("top score %.2f is below the threshold %.2f", 0.21, 0.30)
log.setLevel(logging.DEBUG)
log.debug("now DEBUG records are visible too")

### 5.9 `uuid` and `hashlib` — ids and content hashes

- `uuid.uuid4()` is random: good for request and trace ids. `uuid.uuid5(namespace, name)` is **deterministic**: the same document always gets the same id, which makes re-ingestion idempotent (upsert instead of duplicate). Python 3.14 adds time-ordered `uuid7()`.
- `hashlib.sha256` of a **canonical** JSON (`sort_keys=True`) is a stable cache key: the same request in a different key order gives the same key.
- Never use the built-in `hash()` for anything stored: string hashing is **randomised per process**, as the two fresh interpreters below show.

In [ ]:
import hashlib
import uuid

request_id = uuid.uuid4()
doc_id = uuid.uuid5(uuid.NAMESPACE_URL, "kb://policies/refunds.md")
print("uuid4:", len(str(request_id)), "chars, version", request_id.version, "- different every run")
print("uuid5:", doc_id, "- identical every run")

def cache_key(request):
    canonical = json.dumps(request, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

a = {"model": "gpt-4.1-mini", "temperature": 0, "messages": [{"role": "user", "content": "Refund policy?"}]}
b = {"messages": [{"role": "user", "content": "Refund policy?"}], "temperature": 0, "model": "gpt-4.1-mini"}
c = {**a, "temperature": 0.7}
print("keys :", cache_key(a)[:12], cache_key(b)[:12], cache_key(c)[:12])

env = {k: v for k, v in os.environ.items() if k != "PYTHONHASHSEED"}
cmd = "import hashlib; print(hash('refund policy'), hashlib.sha256(b'refund policy').hexdigest()[:12])"
runs = [subprocess.run([sys.executable, "-c", cmd], env=env, capture_output=True, text=True).stdout.split() for _ in range(2)]
print("hash() in two processes  :", runs[0][0], "vs", runs[1][0])
print("sha256 in two processes  :", runs[0][1], "vs", runs[1][1])
assert cache_key(a) == cache_key(b) != cache_key(c)
assert runs[0][0] != runs[1][0] and runs[0][1] == runs[1][1]

### 5.10 `textwrap`, `dataclasses`, `enum`

`textwrap.dedent` lets you write a prompt template indented inside your code. One subtle trap: **dedent the template first, then fill it in**. If you dedent an f-string, the inserted multi-line text is not indented, so `dedent` finds no common indent and removes nothing.

In [ ]:
from dataclasses import asdict, dataclass, replace
from enum import StrEnum

context = "Refunds are issued within 30 days.\nDigital goods are non-refundable."

def prompt_wrong(q):
    return textwrap.dedent(f"""\
        Answer using only the context.
        Context:
        {context}
        Question: {q}""")

TEMPLATE = textwrap.dedent("""\
    Answer using only the context.
    Context:
    {context}
    Question: {question}""")

print(prompt_wrong("Can I return an e-book?"), "\n---")
print(TEMPLATE.format(context=textwrap.indent(context, "  "), question="Can I return an e-book?"), "\n---")
print("shorten:", textwrap.shorten("A very long retrieved chunk " * 5, width=50, placeholder=" ..."))

class Role(StrEnum):                               # members ARE strings (Python 3.11+)
    SYSTEM = "system"
    USER = "user"

@dataclass(frozen=True)
class GenConfig:
    model: str = "gpt-4.1-mini"
    temperature: float = 0.0

creative = replace(GenConfig(), temperature=0.9)   # frozen: make a modified copy instead of mutating
print(creative, asdict(creative), json.dumps({"role": Role.USER}), Role.USER == "user")
assert TEMPLATE.startswith("Answer") and prompt_wrong("x").startswith("        Answer")

Dataclasses and classes in depth are in [08 · Classes, OOP and dataclasses](08_classes_oop_dataclasses.ipynb); validated data models are [13 · Type hints and Pydantic](13_type_hints_and_pydantic.ipynb).

### 5.11 `argparse` — command-line interfaces (without leaving the notebook)

Evaluation scripts and ingestion jobs take flags. In a notebook, `sys.argv` holds the **kernel's** own arguments, so pass `parse_args` an explicit list. `exit_on_error=False` (3.9+) makes bad input raise `ArgumentError` instead of calling `sys.exit`.

In [ ]:
import argparse

parser = argparse.ArgumentParser(prog="rag-eval", description="Evaluate a retriever.", exit_on_error=False)
parser.add_argument("dataset", help="path to a JSONL eval set")
parser.add_argument("--k", type=int, default=5, help="chunks to retrieve")
parser.add_argument("--model", choices=["gpt-4.1-mini", "gpt-4.1-nano"], default="gpt-4.1-mini")
parser.add_argument("--rerank", action="store_true")

print("kernel's sys.argv[0]:", Path(sys.argv[0]).name)
args = parser.parse_args(["evals.jsonl", "--k", "8", "--rerank"])
print("parsed:", args, "| k is an int:", args.k + 1)
for bad in (["evals.jsonl", "--k", "eight"], ["evals.jsonl", "--model", "gpt-5"]):
    try:
        parser.parse_args(bad)
    except argparse.ArgumentError as e:
        print("error :", e)
print(parser.format_usage().strip())
assert args.k == 8 and args.rerank and args.model == "gpt-4.1-mini"

## Try it yourself

**1.** Find the three most common words (lower-cased, punctuation removed) in the paragraph below using `re.findall` and `Counter`.

In [ ]:
# Solution — try it yourself first, then run this
paragraph = "The refund was late. The refund was partial, and the customer asked: where is the rest of the refund?"
words = re.findall(r"[a-z']+", paragraph.lower())
top3 = Counter(words).most_common(3)
print(top3)
assert top3[0] == ("the", 5) and top3[1] == ("refund", 3)

**2.** Parse the log line `"score=0.82 label=positive latency_ms=431"` into `{"score": 0.82, "label": "positive", "latency_ms": 431}` with **typed** values, using a regex with named groups.

In [ ]:
# Solution — try it yourself first, then run this
line = "score=0.82 label=positive latency_ms=431"

def typed(value):
    for cast in (int, float):
        try:
            return cast(value)
        except ValueError:
            pass
    return value

parsed = {m["key"]: typed(m["value"]) for m in re.finditer(r"(?P<key>\w+)=(?P<value>\S+)", line)}
print(parsed)
assert parsed == {"score": 0.82, "label": "positive", "latency_ms": 431}

**3.** Deduplicate documents by content, ignoring case and extra whitespace: keep the first of each, using a `sha256` of the normalised text.

In [ ]:
# Solution — try it yourself first, then run this
docs = ["Refunds within 30 days.", "refunds   within 30 days.", "Shipping takes 3-5 days.", "REFUNDS WITHIN 30 DAYS."]
seen, unique = set(), []
for d in docs:
    key = hashlib.sha256(" ".join(d.lower().split()).encode("utf-8")).hexdigest()
    if key not in seen:
        seen.add(key)
        unique.append(d)
print(unique)
assert unique == ["Refunds within 30 days.", "Shipping takes 3-5 days."]

In [ ]:
import shutil

sys.path.remove(str(ROOT))                                   # undo our import-path change
for name in [m for m in sys.modules if m.startswith("mytools")]:
    del sys.modules[name]
shutil.rmtree(ROOT, ignore_errors=True)
print("cleaned up:", not ROOT.exists())

## Say it in an interview

- **Import, in 20 seconds:** "Python checks `sys.modules`; if missing it searches `sys.path` in order — script folder, `PYTHONPATH`, stdlib, site-packages — runs the module's top-level code once, caches it, and binds the name. So a local `random.py` shadows the real one, and edits need a reload."
- **`__main__`:** "Running a file sets `__name__` to `'__main__'`; importing sets it to the module name. The guard keeps imports side-effect free — tests import it safely and multiprocessing workers don't re-run the program."
- **Packaging:** "`src/` layout, `pyproject.toml` for declared dependencies, a lock file for exact versions, one venv per project. Pin exactly in applications, use ranges in libraries."
- **Stdlib picks:** `Counter.most_common`, `defaultdict(list)` for grouping, `deque(maxlen=n)` for a sliding window, `OrderedDict` for an LRU, `lru_cache` for pure expensive functions, `itertools.batched` for API batches, `statistics.quantiles` for p95, `re` named groups for parsing, `uuid5` for idempotent ids, `sha256(sort_keys JSON)` for cache keys.
- **Traps:** `groupby` needs sorted input; `random.shuffle` returns `None`; `hash()` changes between processes; naive vs aware datetimes cannot be compared; `datetime.utcnow()` is naive and deprecated; `lru_cache` needs hashable arguments; `dedent` an f-string with multi-line values does nothing.

## Next

- [12 · Decorators and context managers](12_decorators_and_context_managers.ipynb) — `functools.wraps`, `lru_cache` internals, `contextlib`
- [13 · Type hints and Pydantic](13_type_hints_and_pydantic.ipynb)
- [16 · Python gotchas interviewers love](16_python_gotchas_interviewers_love.ipynb)
- DSA uses of these tools: [hashing and sliding window](../03_dsa/03_hashing_and_sliding_window.ipynb) · [stacks and queues](../03_dsa/04_stacks_and_queues.ipynb) · [tries, union-find and LRU cache](../03_dsa/14_tries_union_find_lru_cache.ipynb)
- Production: [caching, exact and semantic](../16_production/03_caching_exact_and_semantic.ipynb) · [latency percentiles and cost](../16_production/05_latency_percentiles_p50_p95_and_cost.ipynb) · [observability, logging and tracing](../16_production/08_observability_logging_and_tracing.ipynb)
- Theory: [python_basics course](../../python_basics/index.html) · [interview bank](../../ai_interview_prep/index.html)